In [1]:
import numpy as np
import sys
from itertools import combinations

In [2]:
pmax = 5

In [3]:
dp = 1.0
pmin = -pmax
np1 = round((pmax-pmin)/dp) + 1

In [4]:
ps_grid = np.zeros((np1**2,3))
pxs, pys = np.meshgrid(np.linspace(pmin,pmax,np1), np.linspace(pmin,pmax,np1))
ps_grid[:,0] = pxs.reshape(np1**2)
ps_grid[:,1] = pys.reshape(np1**2)
ps_grid[:,2] = np.zeros(np1**2)

In [5]:
kes_grid = np.zeros(np1**2)
kes_grid = np.sqrt(ps_grid[:,0]**2+ps_grid[:,1]**2+ps_grid[:,2]**2)

In [6]:
ps = []
kes = []

# Impose ke < pmax
for i in range(len(kes_grid)):
    if kes_grid[i] < pmax+1e-5 and kes_grid[i] > 0 and ps_grid[i,0] >  0:
        ps.append(ps_grid[i])
        kes.append(kes_grid[i])

np2 = len(kes)
ps = np.array(ps)
kes = np.array(kes)

In [7]:
print(f'Number of momentum modes: {np2}')
print('Momentum modes:')
for i in range(np2):
    print(i, ps[i,:])

Number of momentum modes: 35
Momentum modes:
0 [ 1. -4.  0.]
1 [ 2. -4.  0.]
2 [ 3. -4.  0.]
3 [ 1. -3.  0.]
4 [ 2. -3.  0.]
5 [ 3. -3.  0.]
6 [ 4. -3.  0.]
7 [ 1. -2.  0.]
8 [ 2. -2.  0.]
9 [ 3. -2.  0.]
10 [ 4. -2.  0.]
11 [ 1. -1.  0.]
12 [ 2. -1.  0.]
13 [ 3. -1.  0.]
14 [ 4. -1.  0.]
15 [1. 0. 0.]
16 [2. 0. 0.]
17 [3. 0. 0.]
18 [4. 0. 0.]
19 [5. 0. 0.]
20 [1. 1. 0.]
21 [2. 1. 0.]
22 [3. 1. 0.]
23 [4. 1. 0.]
24 [1. 2. 0.]
25 [2. 2. 0.]
26 [3. 2. 0.]
27 [4. 2. 0.]
28 [1. 3. 0.]
29 [2. 3. 0.]
30 [3. 3. 0.]
31 [4. 3. 0.]
32 [1. 4. 0.]
33 [2. 4. 0.]
34 [3. 4. 0.]


In [8]:
def p_spherical(p):
    absp = np.sqrt(np.sum(p*p))
    theta = np.arctan2(np.sqrt(p[0]**2+p[1]**2), p[2])
    phi = np.arctan2(p[1],p[0])
    return absp, theta, phi

# g factor for H_vv
def g_factor(p1, p2, q1, q2):
    absp1, tp1, pp1 = p_spherical(p1)
    absp2, tp2, pp2 = p_spherical(p2)
    absq1, tq1, pq1 = p_spherical(q1)
    absq2, tq2, pq2 = p_spherical(q2)
    fac1 = np.exp(-1j*pq1)*np.sin(tq1/2.)*np.cos(tq2/2.) - np.exp(-1j*pq2)*np.cos(tq1/2.)*np.sin(tq2/2.)
    fac2 = np.exp(1j*pp1)*np.sin(tp1/2.)*np.cos(tp2/2.) - np.exp(1j*pp2)*np.cos(tp1/2.)*np.sin(tp2/2.)
    return 2 * fac1 * fac2

In [9]:
kec_nt = []

for i1 in range(np2):
    p1 = ps[i1,:]
    for i2 in range(i1,np2):
        p2 = ps[i2,:]
        for i3 in range(np2):
            p3 = ps[i3,:]
            for i4 in range(i3,np2):
                p4 = ps[i4,:]
                pdiff = p1+p2-p3-p4
                if np.sum(np.abs(pdiff)) < 1e-7:
                    k1 = np.sqrt(np.dot(p1,p1))
                    k2 = np.sqrt(np.dot(p2,p2))
                    k3 = np.sqrt(np.dot(p3,p3))
                    k4 = np.sqrt(np.dot(p4,p4))
                    gf = g_factor(p1,p2,p3,p4)
                    if np.abs(k1+k2-k3-k4) < 1e-7 and np.abs(gf) > 1e-7:
                        if i1!=i3:
                            #print(i1, i2, i3, i4, p1, p2, p3, p4, p1+p2-p3-p4, np.abs(pdiff))
                            kec_nt.append([i1,i2,i3,i4])

In [10]:
kec_nt = np.array(kec_nt)
Nkec = len(kec_nt)

ninit = np.sort(np.array([0,5,10,25,28,33])) # N=6
#ninit = np.sort(np.array([0,5,10,12,20,25,28,33])) # N=8
#ninit = np.sort(np.array([0,5,8,10,12,20,25,26,28,33])) # N=10

Nn = len(ninit)

pinit = np.sum(ps[ninit], axis=0)
keinit = np.sum(kes[ninit])

In [11]:
print(f'Reference state is {ninit}')
print(f'Reference state total momentum: {pinit}')
print(f'Reference state kinetic energy: {keinit}')

Reference state is [ 0  5 10 25 28 33]
Reference state total momentum: [13.  0.  0.]
Reference state kinetic energy: 23.300723007650674


In [12]:
binom = np.array(list(combinations([i for i in range(Nn)],2)))
Nbinom = len(binom)

In [13]:
def check(state):
    ans = True
    for i in range(len(state)-2):
        if np.var(state[i:i+3]) < 1e-7:
            ans = False
    return ans

In [14]:
def apply(state):
    newstate = []
    for i in range(Nbinom):
        k = np.array([state[binom[i,0]], state[binom[i,1]]])
        for j in range(Nkec):
            if np.sum(np.abs(k-kec_nt[j,:2]))==0:
                state_i = state.copy()
                state_i[binom[i,0]] = kec_nt[j,2]
                state_i[binom[i,1]] = kec_nt[j,3]
                state_i = np.sort(state_i)
                if check(state_i):
                    newstate.append(state_i)
    return np.array(newstate)

In [15]:
nnewstate = 10  # > 1 to enter the while loop
p_states = np.array([ninit])
newstate1 = np.array([np.zeros(Nn), ninit])
trial = 0

while nnewstate > 1:
    print(f'---------------- H^{trial+1} ----------------')
    newstate2 = np.zeros((1,Nn))
    #newstate3 = np.zeros((1,Nn))
    for i in range(1,len(newstate1)):
        newi = apply(newstate1[i])
        if len(newi) > 0:
            newstate2 = np.append(newstate2, newi, axis=0)
       
    newstate1 = np.array([np.zeros(Nn)])
    for j in range(1,len(newstate2)):
        dist = np.sum(np.abs(p_states - newstate2[j]), axis=1)
        if np.min(dist) > 1e-7:
            p_states = np.append(p_states, [newstate2[j].astype(int)], axis=0)
            newstate1 = np.append(newstate1, [newstate2[j]], axis=0)
    nnewstate = len(newstate1)
    print(f'Number of new states(mod flavor choice) at this round: {nnewstate-1}')
    print(f'Number of states(mod flavor choice) visited so far: {len(p_states)}')
        
    trial += 1

---------------- H^1 ----------------
Number of new states(mod flavor choice) at this round: 3
Number of states(mod flavor choice) visited so far: 4
---------------- H^2 ----------------
Number of new states(mod flavor choice) at this round: 5
Number of states(mod flavor choice) visited so far: 9
---------------- H^3 ----------------
Number of new states(mod flavor choice) at this round: 3
Number of states(mod flavor choice) visited so far: 12
---------------- H^4 ----------------
Number of new states(mod flavor choice) at this round: 0
Number of states(mod flavor choice) visited so far: 12


In [16]:
n_states = 0

for i in range(len(p_states)):
    totp = np.sum(ps[p_states[i]], axis=0)
    totke = np.sum(kes[p_states[i]])
    nstate = 2**Nn/4**(Nn-len(set(p_states[i])))
    strs = ' '.join([str(x) for x in p_states[i]]) + '\n'
    n_states += nstate
    if np.sum(np.abs(totp-pinit)) > 1e-7 or np.abs(totke-keinit) > 1e-7:
        print('Total momentum or kinetic energy is not conserved!')

In [17]:
print(f'Number of momentum modes pair with conserved P and E is {len(p_states)}')
print(f'Number of states with conserved P,E, and arbitrary flavor contents is {n_states:.0f}')
bins_visited = np.sort(np.array(list(set(p_states.flatten()))))
print(f'Activated momentum modes:')
print(bins_visited)

Number of momentum modes pair with conserved P and E is 12
Number of states with conserved P,E, and arbitrary flavor contents is 672
Activated momentum modes:
[ 0  1  3  5  8 10 13 22 25 27 28 30 32 33]
